In [1]:
using Plots
using Random
using Statistics

In [27]:
mutable struct Molecular_Dynamics
    # scale
    xL::Float64          # size
    yL::Float64
    time::Float64
    iter::Int
    dt::Float64

    # physical quantities of molecules
    x           # position: -L/2 < x < L/2
    y           # position: -L/2 < y < L/2
    v_x         # velocity of molecules
    v_y

    diameter    # diameter of molecules

    # thermodynamical parameters
    T::Float64              # temperature
    N::Int                  # total number of molecules
    density::Float64        # number density of molecules
    area_fraction::Float64

    # collision data
    collision_data
    collision_molecules

    # subroutines
    _new_collision_data

    function _init_size_and_position(N, density; ϵ=0.01)
        # set molecules at close-packed lattice points
        lattice_constant = sqrt(2.0 / sqrt(3.0) / density)
        scale = Int(round(sqrt(N / 4.0)))
        lattice_x = sqrt(3.0) * lattice_constant
        lattice_y = 2.0 * lattice_constant
        xL = lattice_x * scale
        yL = lattice_y * scale

        x = []
        y = []
        for layer = 1 : 4
            if layer % 2 == 1
                x0 = ϵ
                y0 = ϵ + ((layer - 1) * 0.5) * lattice_constant
            elseif layer % 2 == 0
                x0 = ϵ + 0.5 * lattice_x
                y0 = ϵ + ((layer - 1) * 0.5) * lattice_constant
            end

            for j = 1 : scale
                y_temp = lattice_y * (j - 1) + y0
                if y_temp >= yL
                    break
                else
                    for i = 1 : scale
                        x_temp = lattice_x * (i - 1) + x0
                        if x_temp >= xL
                            break
                        else
                            push!(x, x_temp)
                            push!(y, y_temp)
                        end
                    end
                end
            end
        end

        return xL, yL, x, y
    end

    function _init_velocity(N, T; criterion=3.5)
        # set v_x and v_y according to Maxwell-Boltzmann distribution via Box-Muller algorithm
        v_x = []
        v_y = []
        i = 1
        while i <= N
            box_muller_1 = sqrt(2.0 * T * (-log(rand(Float64))))
            box_muller_2 = 2.0 * π * rand(Float64)
            v_x_candidate = box_muller_1 * cos(box_muller_2)
            v_y_candidate = box_muller_1 * sin(box_muller_2)

            # reject too fast molecules
            if (v_x_candidate ^ 2 + v_y_candidate ^ 2) > T * criterion ^ 2
                continue
            else
                push!(v_x, v_x_candidate)
                push!(v_y, v_y_candidate)
                i += 1
            end
        end

        # correct total momentum to 0
        momentum_x = mean(v_x)
        momentum_y = mean(v_y)
        for i = 1 : N
            v_x[i] -= momentum_x
            v_y[i] -= momentum_y
        end

        # scale velocity to keep formula: T = 1/2 * v^2
        tmp = 0.0
        for i = N
            tmp += v_x[i] ^ 2 + v_y[i] ^ 2
        end
        velocity_temperature = tmp / (2.0 * N)

        scale = sqrt(T / velocity_temperature)
        v_x = v_x .* scale
        v_y = v_y .* scale

        return v_x, v_y
    end

    function _new_collision_data(N, diameter, i, xL, yL, x, y, v_x, v_y)
        time = Inf64
        partner = N
        for j = 1 : N
            if j == i
                continue
            end

            distance_x = x[i] - x[j]
            distance_y = y[i] - y[j]
            relative_velocity_x = v_x[i] - v_x[j]
            relative_velocity_y = v_y[i] - v_y[j]
            # cyclic boundary condition
            distance_x -= round(distance_x / xL) * xL
            distance_y -= round(distance_y / yL) * yL

            inner_product = distance_x * relative_velocity_x + distance_y * relative_velocity_y
            distance_squared = distance_x ^ 2 + distance_y ^ 2
            relative_velocity_squared = relative_velocity_x ^ 2 + relative_velocity_y ^ 2
            if inner_product < 0.0
                discriminator = inner_product ^ 2 - relative_velocity_squared * (distance_squared - diameter ^ 2)
                if discriminator > 0.0
                    collision_time_ij = (-inner_product - sqrt(discriminator)) / relative_velocity_squared
                    if collision_time_ij < time
                        time = collision_time_ij
                        partner = j
                    end
                end
            end
        end

        return time, partner
    end

    function _init_collision_data(N, diameter, xL, yL, x, y, v_x, v_y)
        collision_data = []
        for i = 1 : N
            collision_time, collision_partner = _new_collision_data(N, diameter, i, xL, yL, x, y, v_x, v_y)
            push!(collision_data, [collision_time, collision_partner])
        end

        return collision_data
    end

    function Molecular_Dynamics(T, N, area_fraction; diameter=1.0)
        density = area_fraction * 4.0 / π
        iter = 0
        time = 0.0
        dt = 0.0

        xL, yL, x, y = _init_size_and_position(N, density)
        v_x, v_y = _init_velocity(N, T)
        collision_data = _init_collision_data(N, diameter, xL, yL, x, y, v_x, v_y)
        collision_molecules = (0, 0)

        new(xL, yL, time, iter, dt, x, y, v_x, v_y, diameter, T, N, density, area_fraction, collision_data, collision_molecules, _new_collision_data)
    end
end

function show_parameters(md)
    println("size_x: $(md.xL), size_y: $(md.yL)")
    println("molecular diameter: $(md.diameter)")
    println("thermodynamical parameters:")
    println("temperature: $(md.T), # of molecules: $(md.N), density: $(md.density), area_fraction: $(md.area_fraction)")
end

function solve!(md, iter_max)
    show_parameters(md)
    data = init_data(md, iter_max)

    while md.iter <= iter_max
        dt_and_collision_molecules_update!(md)
        time_evolution!(md)
        velocity_update!(md)
        collision_data_update!(md)
        save_position_data!(md, data)
    end

    plot_data(md, data)
    return data
end

function init_data(md, iter_max)
    data = zeros(md.N, iter_max+2, 2)
    for i = 1 : md.N
        data[i, 1, 1] = md.x[i]
        data[i, 1, 2] = md.y[i]
    end
    return data
end

function dt_and_collision_molecules_update!(md)
    dt = Inf64
    molecule = 0
    for i = 1 : md.N
        if md.collision_data[i][1] < dt
            dt = md.collision_data[i][1]
            molecule = i
        end
    end

    md.dt = dt
    partner = Int(round(md.collision_data[molecule][2]))
    md.collision_molecules = (molecule, partner)
end

function time_evolution!(md)
    md.iter += 1
    md.time += md.dt
    for i = 1 : md.N
        md.collision_data[i][1] -= md.dt

        x_temp = md.x[i] + md.v_x[i] * md.dt
        y_temp = md.y[i] + md.v_y[i] * md.dt
        x_temp -= round(x_temp / md.xL - 0.5) * md.xL       # cyclic boundary condition
        y_temp -= round(y_temp / md.yL - 0.5) * md.yL
        md.x[i] = x_temp
        md.y[i] = y_temp
    end
end

function velocity_update!(md)
    (mol_1, mol_2) = md.collision_molecules

    distance_x = md.x[mol_1] - md.x[mol_2]
    distance_y = md.y[mol_1] - md.y[mol_2]
    distance_x -= round(distance_x / md.xL) * md.xL         # cyclic boundary condition
    distance_y -= round(distance_y / md.yL) * md.yL
    distance = sqrt(distance_x ^ 2 + distance_y ^ 2)
    relative_velocity_x = md.v_x[mol_1] - md.v_x[mol_2]
    relative_velocity_y = md.v_y[mol_1] - md.v_y[mol_2]
    factor = (distance_x * relative_velocity_x + distance_y * relative_velocity_y) / md.diameter
    Δv_x = factor * distance_x / distance
    Δv_y = factor * distance_y / distance

    md.v_x[mol_1] -= Δv_x
    md.v_x[mol_2] += Δv_x
    md.v_y[mol_1] -= Δv_y
    md.v_y[mol_2] += Δv_y
end

function collision_data_update!(md)
    (mol_1, mol_2) = md.collision_molecules
    _collision_data_update!(md, mol_1)
    _collision_data_update!(md, mol_2)

    for i = 1 : md.N
        partner = Int(round(md.collision_data[i][2]))
        if (partner == mol_1) || (partner == mol_2)
            _collision_data_update!(md, i)
        end
    end
end

function _collision_data_update!(md, molecule)
    time, partner = md._new_collision_data(md.N, md.diameter, molecule, md.xL, md.yL, md.x, md.y, md.v_x, md.v_y)
    md.collision_data[molecule] = [time, partner]
end

function save_position_data!(md, data)
    for i = 1 : md.N
        data[i, md.iter+1, 1] = md.x[i]
        data[i, md.iter+1, 2] = md.y[i]
    end
end

function plot_data(md, data)
    plt = scatter()
    for i = 1 : md.N
        scatter!(plt, data[i, :, 1], data[i, :, 2], size=(1050, 900), markeralpha=0.7, markerstrokewidth=0)
    end
    plot(plt)
end

plot_data (generic function with 1 method)

In [11]:
# gas state
T = 5.0
N = 36
area_fraction = 0.3

md = Molecular_Dynamics(T, N, area_fraction)

show_parameters(md)

size_x: 9.034432543914411, size_y: 10.432064122409002
molecular diameter: 1.0
thermodynamical parameters:
temperature: 5.0, # of molecules: 36, density: 0.3819718634205488, area_fraction: 0.3


In [12]:
data = solve!(md, 500)

size_x: 9.034432543914411, size_y: 10.432064122409002
molecular diameter: 1.0
thermodynamical parameters:
temperature: 5.0, # of molecules: 36, density: 0.3819718634205488, area_fraction: 0.3


36×502×2 Array{Float64, 3}:
[:, :, 1] =
 0.01     8.80587    8.76313    8.69573   …  5.98536   5.99521   6.02357
 3.02148  2.92597    2.90886    2.88188      3.76247   3.76696   3.77988
 6.03296  5.92686    5.90786    5.87788      2.02144   2.03564   2.07646
 0.01     0.57095    0.671457   0.669838     1.77457   1.76211   1.72628
 3.02148  3.4253     3.38289    3.31602      8.32206   8.34389   8.4067
 6.03296  5.99178    5.9844     5.97277   …  1.08894   1.06728   1.00496
 0.01     0.0858297  0.0994163  0.120839     1.02156   1.01967   1.01423
 3.02148  3.03345    3.03559    3.03897      7.59214   7.603     7.63425
 6.03296  6.06169    6.06684    6.07496      6.3297    6.32044   6.29382
 1.51574  1.78978    1.83888    1.9163       2.94853   2.94169   2.92201
 4.52722  4.46325    4.45179    4.43372   …  5.68248   5.68238   5.6821
 7.53869  7.51566    7.51154    7.50503      8.07318   8.09877   8.17239
 1.51574  1.60165    1.61704    1.8014       2.50858   2.51337   2.52715
 ⋮           

In [28]:
plot_data(md, data)

Output hidden; open in https://colab.research.google.com to view.

In [14]:
# liquid state
T = 3.0
N = 36
area_fraction = 0.5

md2 = Molecular_Dynamics(T, N, area_fraction)

show_parameters(md2)

size_x: 6.998041357002965, size_y: 8.080642122531591
molecular diameter: 1.0
thermodynamical parameters:
temperature: 3.0, # of molecules: 36, density: 0.6366197723675814, area_fraction: 0.5


In [15]:
data2 = solve!(md2, 500)

size_x: 6.998041357002965, size_y: 8.080642122531591
molecular diameter: 1.0
thermodynamical parameters:
temperature: 3.0, # of molecules: 36, density: 0.6366197723675814, area_fraction: 0.5


36×502×2 Array{Float64, 3}:
[:, :, 1] =
 0.01     6.95967    6.95226    6.94993    …  0.0656754  0.0617691  0.0673719
 2.34268  2.37215    2.37666    2.37808       2.86455    2.84805    2.83961
 4.67536  4.82068    4.84295    4.84994       5.12659    5.13171    5.13432
 0.01     0.0897799  0.102008   0.105848      0.0255703  0.0425379  0.0512112
 2.34268  2.33993    2.33951    2.33937       2.76612    2.76117    2.75864
 4.67536  4.69271    4.69537    4.69621    …  4.25856    4.25174    4.24826
 0.01     6.97653    6.97169    6.97018       6.53255    6.52078    6.51477
 2.34268  2.37501    2.37997    2.38152       1.6727     1.67084    1.6699
 4.67536  4.80407    4.8238     4.81711       3.60676    3.6093     3.6106
 1.17634  0.994111   0.96618    0.957409      1.26975    1.26818    1.26738
 3.50902  3.50272    3.50176    3.50146    …  3.96595    3.96941    3.97119
 5.8417   5.60613    5.57002    5.55868       6.05437    6.06538    6.06341
 1.17634  1.21571    1.22175    1.22364       

In [29]:
plot_data(md2, data2)

Output hidden; open in https://colab.research.google.com to view.

In [17]:
# solid state
T = 1.0
N = 36
area_fraction = 0.7

md3 = Molecular_Dynamics(T, N, area_fraction)

show_parameters(md3)

size_x: 5.914424427637176, size_y: 6.829389070796044
molecular diameter: 1.0
thermodynamical parameters:
temperature: 1.0, # of molecules: 36, density: 0.8912676813146139, area_fraction: 0.7


In [19]:
data3 = solve!(md3, 500)

size_x: 5.914424427637176, size_y: 6.829389070796044
molecular diameter: 1.0
thermodynamical parameters:
temperature: 1.0, # of molecules: 36, density: 0.8912676813146139, area_fraction: 0.7


36×502×2 Array{Float64, 3}:
[:, :, 1] =
 0.01      5.87549    5.87059    5.86009   …  5.89173   5.89327   5.89535
 1.98147   1.94377    1.94       1.93191      2.18057   2.17983   2.17915
 3.95295   3.9879     3.99139    3.99889      3.94527   3.9452    3.94511
 0.01      5.86844    5.86284    5.85083      5.68602   5.68617   5.68636
 1.98147   1.96795    1.96659    1.96369      1.59775   1.59637   1.5945
 3.95295   3.93551    3.93376    3.93002   …  3.61001   3.60755   3.60423
 0.01      0.0536514  0.0580152  0.067382     0.257721  0.254787  0.250833
 1.98147   2.01039    2.01328    2.01948      2.18036   2.18288   2.18627
 3.95295   3.98532    3.97657    3.95779      4.05164   4.05484   4.05915
 0.995737  0.986176   0.98522    0.983169     0.826658  0.826011  0.825139
 2.96721   2.98789    2.98996    2.99439   …  3.07151   3.0694    3.06657
 4.93869   4.97432    4.97788    4.98552      4.87209   4.8706    4.8686
 0.995737  0.950394   0.945861   0.95844      0.733748  0.729812  0.7245

In [30]:
plot_data(md3, data3)

Output hidden; open in https://colab.research.google.com to view.